# Titanic - Train on Colab GPU

Mounts Google Drive and trains the `TabularResNet` using the code + data you uploaded.

**Upload this folder to your Drive at `MyDrive/IAI_TITANIC/`:**
```
MyDrive/IAI_TITANIC/
+-- src/                 (preprocessing.py, model.py, __init__.py)
+-- train.py
+-- prepare_data.py
+-- requirements.txt
+-- data/train.csv       (already-fetched data)
+-- artifacts/           (OPTIONAL: splits.npz + preprocessor.joblib from local `python prepare_data.py`)
```

Set the GPU runtime first: **Runtime -> Change runtime type -> GPU**.

## 1. Mount Drive + enter project

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

PROJECT = '/content/drive/MyDrive/IAI_TITANIC'

Mounted at /content/drive


In [2]:
%cd $PROJECT
!pip install -q torch scikit-learn pandas numpy joblib optuna

/content/drive/MyDrive/IAI_TITANIC


## 2. Training phase + hyperparameters tuning

In [14]:
import optuna
from train import train

def objective(trial):
    # הגדרת מרחב החיפוש להיפר-פרמטרים
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.1, 0.3)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    # הרצת האימון עם הפרמטרים של ה-Trial הנוכחי
    res = train(
        splits_path='artifacts/splits.npz',
        preprocessor_path='artifacts/preprocessor.joblib',
        out_dir=f'artifacts/trial_{trial.number}',
        epochs=100, # עדיף פחות אופוקים לטובת חיפוש מהיר
        patience=10,
        device='cuda',
        lr=lr,
        dropout=dropout,
        batch_size=batch_size,
        hidden_dim = trial.suggest_categorical("hidden_dim", [16, 32, 64]),
        n_blocks   = trial.suggest_int("n_blocks", 1, 3)
    )

    # אנחנו רוצים למקסם את ה-Validation F1 או Accuracy
    return res['best_metrics']['roc_auc']

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best trial:")
print(study.best_trial.value)
print(study.best_trial.params)

[I 2026-06-17 13:29:23,296] A new study created in memory with name: no-name-cc7d1a86-737c-48f0-93ae-edb2cc867deb


[data] loaded precomputed splits from artifacts/splits.npz (19 features)
[setup] device=cuda | train=712 val=179 | features=19
epoch   1 | train_loss 0.8017 | val_loss 0.8262 | acc 0.704 | f1 0.595 | auc 0.764
epoch   2 | train_loss 0.6469 | val_loss 0.7754 | acc 0.726 | f1 0.675 | auc 0.804
epoch   3 | train_loss 0.5911 | val_loss 0.6822 | acc 0.788 | f1 0.716 | auc 0.837
epoch   4 | train_loss 0.5575 | val_loss 0.6068 | acc 0.827 | f1 0.744 | auc 0.850
epoch   5 | train_loss 0.5318 | val_loss 0.5722 | acc 0.827 | f1 0.752 | auc 0.857
epoch   6 | train_loss 0.5288 | val_loss 0.5571 | acc 0.832 | f1 0.766 | auc 0.860
epoch   7 | train_loss 0.5170 | val_loss 0.5485 | acc 0.827 | f1 0.777 | auc 0.865
epoch   8 | train_loss 0.4945 | val_loss 0.5449 | acc 0.838 | f1 0.785 | auc 0.868
epoch   9 | train_loss 0.4990 | val_loss 0.5451 | acc 0.832 | f1 0.795 | auc 0.869
epoch  10 | train_loss 0.4916 | val_loss 0.5468 | acc 0.832 | f1 0.773 | auc 0.868
epoch  11 | train_loss 0.4907 | val_loss 0.

[I 2026-06-17 13:29:25,632] Trial 0 finished with value: 0.8675889328063242 and parameters: {'lr': 0.0006160856099580579, 'dropout': 0.10793685725765706, 'batch_size': 64, 'hidden_dim': 64, 'n_blocks': 1}. Best is trial 0 with value: 0.8675889328063242.


epoch  18 | train_loss 0.4574 | val_loss 0.5656 | acc 0.816 | f1 0.781 | auc 0.858
[early-stop] no val-loss improvement for 10 epochs.

[done] best epoch 8 | val acc 0.838 | f1 0.785 | auc 0.868
[saved] artifacts/trial_0/model.pt
[saved] artifacts/trial_0/preprocessor.joblib
[saved] artifacts/trial_0/metadata.json
[data] loaded precomputed splits from artifacts/splits.npz (19 features)
[setup] device=cuda | train=712 val=179 | features=19
epoch   1 | train_loss 0.6821 | val_loss 0.8069 | acc 0.771 | f1 0.672 | auc 0.800
epoch   2 | train_loss 0.5529 | val_loss 0.7066 | acc 0.816 | f1 0.736 | auc 0.851
epoch   3 | train_loss 0.5387 | val_loss 0.6159 | acc 0.804 | f1 0.762 | auc 0.870
epoch   4 | train_loss 0.4947 | val_loss 0.5611 | acc 0.827 | f1 0.763 | auc 0.865
epoch   5 | train_loss 0.5038 | val_loss 0.5616 | acc 0.832 | f1 0.789 | auc 0.859
epoch   6 | train_loss 0.4971 | val_loss 0.5738 | acc 0.816 | f1 0.740 | auc 0.853
epoch   7 | train_loss 0.4850 | val_loss 0.5639 | acc 0.821

[I 2026-06-17 13:29:27,623] Trial 1 finished with value: 0.8650856389986825 and parameters: {'lr': 0.0036330243284996237, 'dropout': 0.21373524145056175, 'batch_size': 64, 'hidden_dim': 32, 'n_blocks': 2}. Best is trial 0 with value: 0.8675889328063242.


...

epoch  20 | train_loss 0.5270 | val_loss 0.5611 | acc 0.827 | f1 0.774 | auc 0.858
[early-stop] no val-loss improvement for 10 epochs.

[done] best epoch 10 | val acc 0.844 | f1 0.763 | auc 0.877
[saved] artifacts/trial_17/model.pt
[saved] artifacts/trial_17/preprocessor.joblib
[saved] artifacts/trial_17/metadata.json
[data] loaded precomputed splits from artifacts/splits.npz (19 features)
[setup] device=cuda | train=712 val=179 | features=19
epoch   1 | train_loss 0.8798 | val_loss 0.8594 | acc 0.648 | f1 0.364 | auc 0.488
epoch   2 | train_loss 0.8346 | val_loss 0.8401 | acc 0.654 | f1 0.392 | auc 0.615
epoch   3 | train_loss 0.8173 | val_loss 0.8146 | acc 0.749 | f1 0.609 | auc 0.708
epoch   4 | train_loss 0.7940 | val_loss 0.7948 | acc 0.749 | f1 0.681 | auc 0.779
epoch   5 | train_loss 0.7757 | val_loss 0.7768 | acc 0.765 | f1 0.712 | auc 0.800
epoch   6 | train_loss 0.7553 | val_loss 0.7582 | acc 0.777 | f1 0.718 | auc 0.820
epoch   7 | train_loss 0.7233 | val_loss 0.7406 | acc 0

[I 2026-06-17 13:33:26,800] Trial 18 finished with value: 0.8765480895915678 and parameters: {'lr': 0.00018228191104611774, 'dropout': 0.1330300338062642, 'batch_size': 32, 'hidden_dim': 16, 'n_blocks': 2}. Best is trial 15 with value: 0.8845849802371543.


epoch  59 | train_loss 0.4858 | val_loss 0.5442 | acc 0.838 | f1 0.791 | auc 0.874
epoch  60 | train_loss 0.4979 | val_loss 0.5434 | acc 0.844 | f1 0.794 | auc 0.874
[early-stop] no val-loss improvement for 10 epochs.

[done] best epoch 50 | val acc 0.838 | f1 0.782 | auc 0.877
[saved] artifacts/trial_18/model.pt
[saved] artifacts/trial_18/preprocessor.joblib
[saved] artifacts/trial_18/metadata.json
[data] loaded precomputed splits from artifacts/splits.npz (19 features)
[setup] device=cuda | train=712 val=179 | features=19
epoch   1 | train_loss 0.8814 | val_loss 0.7571 | acc 0.709 | f1 0.658 | auc 0.778
epoch   2 | train_loss 0.6928 | val_loss 0.6671 | acc 0.777 | f1 0.737 | auc 0.819
epoch   3 | train_loss 0.6494 | val_loss 0.6031 | acc 0.788 | f1 0.753 | auc 0.855
epoch   4 | train_loss 0.5870 | val_loss 0.5656 | acc 0.810 | f1 0.770 | auc 0.879
epoch   5 | train_loss 0.5789 | val_loss 0.5465 | acc 0.832 | f1 0.758 | auc 0.874
epoch   6 | train_loss 0.5828 | val_loss 0.5462 | acc 0

[I 2026-06-17 13:33:35,331] Trial 19 finished with value: 0.8837944664031622 and parameters: {'lr': 0.0012419761501686625, 'dropout': 0.23514104638420785, 'batch_size': 16, 'hidden_dim': 16, 'n_blocks': 3}. Best is trial 15 with value: 0.8845849802371543.


epoch  20 | train_loss 0.5063 | val_loss 0.5669 | acc 0.821 | f1 0.750 | auc 0.859
[early-stop] no val-loss improvement for 10 epochs.

[done] best epoch 10 | val acc 0.855 | f1 0.794 | auc 0.884
[saved] artifacts/trial_19/model.pt
[saved] artifacts/trial_19/preprocessor.joblib
[saved] artifacts/trial_19/metadata.json
Best trial:
0.8845849802371543
{'lr': 0.00018345795913905782, 'dropout': 0.18326425481671912, 'batch_size': 16, 'hidden_dim': 16, 'n_blocks': 3}


## 3. Check saved artifacts

Everything is written back to `artifacts/` on your Drive (survives runtime shutdown).

In [15]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import numpy as np

# 1. הצגת הפרמטרים שנבחרו
with open('artifacts/metadata.json', 'r') as f:
    metadata = json.load(f)
print("Training Output:", json.dumps(metadata, indent=2))

# 2. אם הרצנו Optuna, נציג את חשיבות ההיפר-פרמטרים
try:
    from optuna.visualization import plot_param_importances
    fig = plot_param_importances(study)
    fig.show()
except NameError:
    pass

# הוספת קוד לטעינת ה-Validation Set והפקת Confusion Matrix + ROC Curve
# (מצריך לטעון את המודל מ-artifacts/best_model.pt ולהעביר עליו את נתוני ה-Val)

Training Output: {
  "config": {
    "data_path": "data/train.csv",
    "out_dir": "artifacts",
    "splits_path": "artifacts/splits.npz",
    "preprocessor_path": "artifacts/preprocessor.joblib",
    "epochs": 200,
    "patience": 20,
    "batch_size": 32,
    "lr": 0.001,
    "weight_decay": 0.01,
    "hidden_dim": 32,
    "n_blocks": 1,
    "dropout": 0.3,
    "val_size": 0.2,
    "seed": 42,
    "device": "cuda"
  },
  "device": "cuda",
  "feature_names": [
    "Age",
    "FareLog",
    "FamilySize",
    "Sex_female",
    "Sex_male",
    "Embarked_C",
    "Embarked_Q",
    "Embarked_S",
    "Title_Master",
    "Title_Miss",
    "Title_Mr",
    "Title_Mrs",
    "Title_Rare",
    "Pclass_1",
    "Pclass_2",
    "Pclass_3",
    "IsAlone",
    "HasCabin",
    "IsChild"
  ],
  "best_epoch": 14,
  "val_metrics": {
    "loss": 0.5358147443006824,
    "accuracy": 0.8547486033519553,
    "f1": 0.796875,
    "macro_f1": 0.8419157608695652,
    "roc_auc": 0.8718050065876153,
    "best_thresh"

## Identifying the Best Model from Optuna

In [16]:
if 'study' in locals():
    print("Best trial found by Optuna:")
    print(f"  Value (ROC AUC): {study.best_trial.value:.4f}")
    print("  Parameters:")
    for key, value in study.best_trial.params.items():
        print(f"    {key}: {value}")
else:
    print("Optuna study object not found. Please run the Optuna cell (S82OboFyWVCM) first.")

Best trial found by Optuna:
  Value (ROC AUC): 0.8846
  Parameters:
    lr: 0.00018345795913905782
    dropout: 0.18326425481671912
    batch_size: 16
    hidden_dim: 16
    n_blocks: 3
